# Semi-Supervised Learning & Deep Clustering

## Imports

In [1]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
from sklearn.cluster import KMeans
import random

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## Data Loading

In [ ]:
# Create labeled and unlabeled subsets
labeled_indices = []
targets = np.array([train_dataset[i][1] for i in range(len(train_dataset))])
for cls in range(10):
    cls_indices = np.where(targets == cls)[0]
    labeled_indices.extend(np.random.choice(cls_indices, 100, replace=False))
unlabeled_indices = list(set(range(len(train_dataset))) - set(labeled_indices))

# TODO: create Subsets
labeled_dataset = Subset(train_dataset, labeled_indices)  
unlabeled_dataset = Subset(train_dataset, unlabeled_indices)  

# TODO: create loaders
labeled_loader = DataLoader(labeled_dataset, batch_size=64, shuffle=True)   
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True)   
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [3]:
print(f"Number of labeled samples: {len(labeled_dataset)}")
print(f"Number of unlabeled samples: {len(unlabeled_dataset)}")

Number of labeled samples: 1000
Number of unlabeled samples: 59000


## $\Pi$-Model

### Model Definition

**$\Pi$-Model** is a classic architecture in **Semi-Supervised Learning.** It relies in the concept of **consistency regularization:** it assumes that if the same input is given twice with slight stochastic changes, the output should remain roughly the same.

The stochastic permutation is handled by **Dropout:** because Dropout randomly deactivates neurons, the model's internal path is different every time we run a forward pass.

In [ ]:
class PiModelCNN(nn.Module):
    def __init__(self):
        super(PiModelCNN, self).__init__()
        # TODO: conv layer with 1 input channel and 32 output channels
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)     
        # TODO: max pooling layer
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)     
        # TODO: dropout layer, use p=0.25
        self.dropout1 = nn.Dropout2d(p=0.25)  
        # TODO: conv layer with 32 input channels and 64 output channels
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)     
        # TODO: max pooling layer
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)     
        # TODO: dropout layer, use p=0.25
        self.dropout2 = nn.Dropout2d(p=0.25)  
        
        self.fc1 = nn.Linear(64 * 7 * 7, 64)
        
        # TODO: dropout layer, use p=0.5
        self.dropout3 = nn.Dropout(p=0.5)  
        # TODO: linear layer, 64 inputs, 10 outputs
        self.fc2 = nn.Linear(64, 10)       

    def forward(self, x):
        # TODO: pass x through your layers.
        # Apply a relu activation after conv1, conv2, and fc1
        x = nn.functional.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.dropout1(x)
        
        x = nn.functional.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.dropout2(x)
        
        # Flatten before linear layers
        x = x.view(-1, 64 * 7 * 7)
        
        x = nn.functional.relu(self.fc1(x))
        x = self.dropout3(x)
        
        x = self.fc2(x)
        
        # Note: Depending on your exact CrossEntropy setup, you usually 
        # return raw logits. We leave softmax here per the template instructions.
        return torch.softmax(x, dim=1)

### Training Loop

When training a $\Pi$-Model we try at the same time to teach the model how to predict **similar outputs for slightly different inputs** and how to correctly classify the **few labeled samples** we dispose of.

This means that the Loss Function is composed of **two terms:** the first one refers to the **Supervised Loss** (how good the model is at classifying labeled data), and the second one refers to the **Consistency Loss** (how good the model is at producing similar outputs given similar input).

The final loss is given by:

$$Loss = \text{Supervised Loss} + \lambda(t) \cdot \text{Consistency Loss}$$

where **$\lambda$(t)** is used to weight more one loss over the other, preventing one of the two being suffocated.

In [ ]:
# Initialize model, loss, and optimizer
pi_model = PiModelCNN().cuda()       # TODO: instantiate the Pi Model and move to device
criterion_ce = nn.CrossEntropyLoss()   # TODO: choose the right loss for classification
criterion_mse = nn.MSELoss()  # TODO: mse loss
optimizer_pi = optim.Adam(pi_model.parameters(), lr=0.001)

In [ ]:
def train_pi_model(model, labeled_loader, unlabeled_loader, epochs=300):
    model.train()

    # Define the weight of criterion_mse against criterion_ce
    consistency_weight = 10.0

    for epoch in range(epochs):
        total_loss = 0
        for (labeled_data, labeled_targets), (unlabeled_data, _) in zip(labeled_loader, unlabeled_loader):
            labeled_data, labeled_targets = labeled_data.cuda(), labeled_targets.cuda()
            unlabeled_data = unlabeled_data.cuda()

            # Forward pass for labeled data
            # TODO: compute prediction for labeled data and compute the loss
            sup_preds = model(labeled_data)
            sup_loss = criterion_ce(sup_preds, labeled_targets)

            # Two forward passes for unlabeled data with different dropout
            model.train()
            unlabeled_preds1 = model(unlabeled_data)   # TODO
            unlabeled_preds2 = model(unlabeled_data)   # TODO

            # Compute consistency loss
            cons_loss = criterion_mse(unlabeled_preds1, unlabeled_preds2)  # TODO

            # Combine losses
            loss = sup_loss + consistency_weight * cons_loss   # TODO

            # Backward pass
            optimizer_pi.zero_grad()
            loss.backward()
            optimizer_pi.step()

            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f'Epoch {epoch}, Loss: {total_loss / len(labeled_loader):.4f}')

In [ ]:
# Train the Π-Model
train_pi_model(pi_model, labeled_loader, unlabeled_loader, epochs=300)

## Deep Clustering

**Deep Clustering** is a model typically used to solve Unsupervised Learning. Here we adapt it for Semi-Supervised Learning.

This model focuses on the **structure **of data: it tries to force unlabeled data into **distinct, tight groups**, based on similarities or dissimilarities between the samples.

### Model Definition

In [ ]:
class ClusteringLayer(nn.Module):
    def __init__(self, n_clusters=10):
        super(ClusteringLayer, self).__init__()
        # Number of clusters we have to find
        self.n_clusters = n_clusters  # TODO

        # Coordinates of the centers of the 10 clusters
        self.cluster_centers = nn.Parameter(torch.randn(n_clusters, 10))

    def forward(self, x):
        # Compute the Squared Euclidean Distance between the input data sample x
        # and every cluster center. 
        # x shape: (batch_size, 10), centers shape: (n_clusters, 10)
        d = torch.sum((x.unsqueeze(1) - self.cluster_centers.unsqueeze(0)) ** 2, dim=2)  # TODO

        # Converts a distance into a similarity score between 0 and 1
        # using a Student's t-distribution
        q = 1.0 / (1.0 + d)

        # Sharpening the similarity scores
        q = q ** 2  # TODO: square q

        # Normalize (ensure scores sum up to 1)
        q = q / torch.sum(q, dim=1, keepdim=True)  # TODO

        return q

class DeepClusteringModel(nn.Module):
    def __init__(self):
        super(DeepClusteringModel, self).__init__()

        # Define the feature extractor
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

        # Define classifier
        self.classifier = nn.Linear(10, 10)  # TODO: linear layer, 10 inputs, 10 outputs

        # Define clustering
        self.clustering = ClusteringLayer(n_clusters=10)  # TODO: instantiate clustering layer

    def forward(self, x, mode='both'):
        features = self.encoder(x)
        if mode == 'classifier':
            return torch.softmax(self.classifier(features), dim=1)
        elif mode == 'clustering':
            return self.clustering(features)
        else:
            return torch.softmax(self.classifier(features), dim=1), self.clustering(features)

### Training Loop

Training involves two phases:


1.   **Supervised Pre-training:** we use the labeled data we have to teach the model an **initial representation** of the data. In this way the model should be able to extract relevant features to distinguish between different classes.
We then use these features to **initialize the centers of the clusters.** Instead of assigning random values, we start from a **solid baseline,** using as initial values the center of the clusters of features extracted by the model.
2.   **Actual Training:** in this phase we teach the model how to correctly **classify labeled samples** and, at the same time, how to assign each unlabeled sample to the **correct cluster.**
This last step is done through a technique known as **self-training:** given a unlabeled samples, the model predicts a **soft guess q.** This soft guess is then **sharpened**, producing a target distribution p, which is a more confident prediction. The goal is then to **minimize the difference** between q and p, pushing q toward p.

Since q and p are distributions telling the probability of the given sample to belong to each cluster, the loss between them is computed using the **Kullback-Leibler (KL) Divergence:**

$$KL(P || Q) = \sum P(i) \log \frac{P(i)}{Q(i)}$$

This metric measures the **distance **between two probability distributions or, more precisely, how well a given probability q approximates a target probability p.


The final loss function is given by a **weighted sum** between the classification loss and the KL Divergence.

In [ ]:
# Initialize model and optimizer
dc_model = DeepClusteringModel().cuda() # TODO: remember to move model to right device
optimizer_dc = optim.Adam(dc_model.parameters(), lr=0.001)

In [ ]:
def pretrain_dc_model(model, labeled_loader, epochs=30):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for data, targets in labeled_loader:
            data, targets = data.cuda(), targets.cuda()

            preds = model(data, mode='classifier')  # TODO: compute predictions
            loss = criterion_ce(preds, targets)   # TODO: compute loss

            # TODO: backward pass
            optimizer_dc.zero_grad()
            loss.backward()
            optimizer_dc.step()
            
            total_loss += loss.item()
        print(f'Pretrain Epoch {epoch}, Loss: {total_loss / len(labeled_loader):.4f}')

def initialize_clusters(model, labeled_loader):
    model.eval()
    features = []
    with torch.no_grad():
        for data, _ in labeled_loader:
            data = data.cuda()

            feat = model.encoder(data) # TODO: use your model to extract features from data
            features.append(feat.cpu().numpy())

    features = np.concatenate(features, axis=0)

    kmeans = KMeans(n_clusters=10, n_init=20, random_state=42) # TODO: instantiate KMeans with 10 clusters

    # TODO: fit kmeans model
    kmeans.fit(features)

    model.clustering.cluster_centers.data = torch.tensor(kmeans.cluster_centers_).cuda()

def train_dc_model(model, labeled_loader, unlabeled_loader, epochs=300):
    model.train()
    clustering_weight = 0.1
    # KLDivLoss expects log-probabilities for the input
    criterion_kl = nn.KLDivLoss(reduction='batchmean') # TODO

    for epoch in range(epochs):
        total_loss = 0
        for (labeled_data, labeled_targets), (unlabeled_data, _) in zip(labeled_loader, unlabeled_loader):
            labeled_data, labeled_targets = labeled_data.cuda(), labeled_targets.cuda()
            unlabeled_data = unlabeled_data.cuda()

            # Forward pass for labeled data
            sup_preds = model(labeled_data, mode='classifier')
            sup_loss = criterion_ce(sup_preds, labeled_targets) # TODO

            # Forward pass for unlabeled data
            _, cluster_probs = model(unlabeled_data, mode='both') # TODO

            # Compute target distribution (sharpened)
            p = cluster_probs ** 2 / torch.sum(cluster_probs, dim=0)
            p = p / torch.sum(p, dim=1, keepdim=True)

            # KL divergence expects log(q) and p
            cluster_loss = criterion_kl(torch.log(cluster_probs + 1e-8), p) # TODO

            # Combine losses
            loss = sup_loss + clustering_weight * cluster_loss # TODO

            # TODO: backward pass
            optimizer_dc.zero_grad()
            loss.backward()
            optimizer_dc.step()

            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f'Epoch {epoch}, Loss: {total_loss / len(labeled_loader):.4f}')

In [ ]:
pretrain_dc_model(dc_model, labeled_loader, epochs=30)
initialize_clusters(dc_model, labeled_loader)
train_dc_model(dc_model, labeled_loader, unlabeled_loader, epochs=300)

## Evaluation and Model Comparison

In [ ]:
def evaluate_model(model, test_loader, mode='classifier'):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, targets in test_loader:
            data, targets = data.cuda(), targets.cuda()
            # Check if model is DeepClusteringModel (supports mode) or PiModelCNN (does not)
            if isinstance(model, DeepClusteringModel):
                preds = model(data, mode=mode)  # TODO
            else:  # PiModelCNN
                preds = model(data)  # TODO
            _, predicted = torch.max(preds, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    return 100 * correct / total

# [Supervised Baseline Training code provided in your template runs here]

# Evaluate all models
pi_accuracy = evaluate_model(pi_model, test_loader)
dc_accuracy = evaluate_model(dc_model, test_loader)
sup_accuracy = evaluate_model(sup_model, test_loader)

print(f'Π-Model Test Accuracy: {pi_accuracy:.2f}%')
print(f'Deep Clustering Test Accuracy: {dc_accuracy:.2f}%')
print(f'Supervised Baseline Test Accuracy: {sup_accuracy:.2f}%')